# 01. Data Preparation

## Goal

Prepare labeled ToxiCR comments for toxicity classification models: clean the
text, remove invalid records, examine class balance, and save the processed
splits.


## Imports and Data Paths


In [ ]:
import csv
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")
LEXICON_DIR = Path("data/lexicons")
OUTPUT_DIR = Path("data/preprocessed")


## Text Cleaning Rules

Cleaning removes links and email addresses, normalizes apostrophes and
contractions, restores obfuscated profanity, and reduces artificial character
repetition. Technical notation is retained because it carries meaning in code
review comments.


In [ ]:
URL_EMAIL_RE = re.compile(
    r"https?://\S+|www\.\S+|\b[\w.+-]+@[\w-]+\.[\w.-]+\b",
    re.IGNORECASE,
)
MULTISPACE_RE = re.compile(r"\s+")
APOSTROPHE_RE = re.compile(r"[’`´ʹʻ]")
REPEAT_CHAR_RE = re.compile(r"(?i)([a-z])\1{2,}")
NON_TEXT_RE = re.compile(
    r"[^a-z0-9_ \t\n\.,${}\-#+:;/=<>!%*'\"]",
    re.IGNORECASE,
)

CONTRACTIONS = {
    "can't": "can not",
    "won't": "will not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "shouldn't": "should not",
    "isn't": "is not",
    "aren't": "are not",
    "weren't": "were not",
    "wasn't": "was not",
    "i'm": "i am",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "what's": "what is",
    "who's": "who is",
    "let's": "let us",
}


In [ ]:
def load_lexicon(path: Path) -> set[str]:
    with path.open(encoding="utf-8") as file:
        return {line.strip().lower() for line in file if line.strip()}


def normalize_unicode(text: str) -> str:
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    return APOSTROPHE_RE.sub("'", text)


def expand_contractions(text: str) -> str:
    for contraction, expanded in CONTRACTIONS.items():
        text = re.sub(rf"(?i)\b{contraction}\b", expanded, text)
    return text


def clean_text(text: str, profane_words: set[str], programming_words: set[str]) -> str:
    if not isinstance(text, str):
        return ""

    text = normalize_unicode(text).strip().lower()
    text = expand_contractions(URL_EMAIL_RE.sub(" ", text))

    for word in profane_words:
        pattern = r"(?i)" + r"".join(f"{character}[^a-z0-9]{{0,2}}" for character in word)
        text = re.sub(pattern, word, text)

    text = NON_TEXT_RE.sub(" ", text)
    tokens = []
    for token in re.split(r"(\s+)", text):
        if token.isspace():
            tokens.append(token)
        elif any(character.isdigit() or character in "_./#-=%*()[]{}" for character in token):
            tokens.append(token)
        elif token in programming_words:
            tokens.append(token)
        else:
            tokens.append(REPEAT_CHAR_RE.sub(r"\1\1", token))

    return MULTISPACE_RE.sub(" ", "".join(tokens)).strip()


## Loading and Inspecting Raw Data


In [ ]:
def load_dataset(path: Path) -> pd.DataFrame:
    data = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    required_columns = {"message", "is_toxic"}
    if not required_columns.issubset(data.columns):
        raise ValueError(f"File {path} is missing the required columns {required_columns}.")
    return data


raw_train = load_dataset(RAW_DIR / "train.csv")
raw_test = load_dataset(RAW_DIR / "test.csv")

pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(raw_train), len(raw_test)],
        "non_toxic": [(raw_train["is_toxic"] == 0).sum(), (raw_test["is_toxic"] == 0).sum()],
        "toxic": [(raw_train["is_toxic"] == 1).sum(), (raw_test["is_toxic"] == 1).sum()],
    }
)


## Cleaning and Saving Processed Splits


In [ ]:
profane_words = load_lexicon(LEXICON_DIR / "profane-words.txt")
programming_words = load_lexicon(LEXICON_DIR / "programming_keywords.txt")


def preprocess(data: pd.DataFrame) -> pd.DataFrame:
    cleaned = data.copy()
    cleaned["message"] = cleaned["message"].fillna("").astype(str)
    cleaned["message"] = cleaned["message"].map(
        lambda text: clean_text(text, profane_words, programming_words)
    )
    cleaned = cleaned[cleaned["message"].str.len() > 0]
    cleaned = cleaned.dropna(subset=["is_toxic"])
    cleaned = cleaned.drop_duplicates(subset=["message", "is_toxic"])
    cleaned["is_toxic"] = cleaned["is_toxic"].astype(int)
    return cleaned


clean_train = preprocess(raw_train)
clean_test = preprocess(raw_test)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean_train.to_csv(OUTPUT_DIR / "clean_train.csv", sep=";", index=False, quoting=csv.QUOTE_ALL)
clean_test.to_csv(OUTPUT_DIR / "clean_test.csv", sep=";", index=False, quoting=csv.QUOTE_ALL)


## Validating the Processing Result


In [ ]:
statistics = pd.DataFrame(
    {
        "split": ["train", "test"],
        "raw_rows": [len(raw_train), len(raw_test)],
        "cleaned_rows": [len(clean_train), len(clean_test)],
        "non_toxic": [(clean_train["is_toxic"] == 0).sum(), (clean_test["is_toxic"] == 0).sum()],
        "toxic": [(clean_train["is_toxic"] == 1).sum(), (clean_test["is_toxic"] == 1).sum()],
    }
)
statistics


In [ ]:
examples = raw_train["message"].dropna().head(5)
pd.DataFrame(
    {
        "before_cleaning": examples,
        "after_cleaning": examples.map(
            lambda text: clean_text(text, profane_words, programming_words)
        ),
    }
)


## Conclusion

The source comments have been cleaned and saved under `data/preprocessed/`.
The processed splits are ready for comparison of the classical model and the
transformer-based model in the following notebooks.
